In [1]:
import pickle
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

In [2]:
# extra work as x test is corrupted
import pandas as pd
from sklearn.model_selection import train_test_split

# Load raw data
df = pd.read_csv("../../../../data/creditcard.csv")

# Separate features and target
X = df.drop("Class", axis=1)
y = df["Class"]

# Split — all 4 files created from the SAME split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Save all 4 files
X_train.to_pickle("../../../../data/processed/X_train.pkl")
X_test.to_pickle("../../../../data/processed/X_test.pkl")
y_train.to_pickle("../../../../data/processed/y_train.pkl")
y_test.to_pickle("../../../../data/processed/y_test.pkl")

print("All files regenerated!")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")


# loading the preprocessed data from pkl

import pandas as pd

X_train = pd.read_pickle("../../../../data/processed/X_train.pkl")
X_test  = pd.read_pickle("../../../../data/processed/X_test.pkl")
y_train = pd.read_pickle("../../../../data/processed/y_train.pkl")
y_test  = pd.read_pickle("../../../../data/processed/y_test.pkl")


All files regenerated!
X_train: (227845, 30), X_test: (56962, 30)
y_train: (227845,), y_test: (56962,)


In [3]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    # For ROC-AUC
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
    else:
        y_prob = y_pred

    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

    print(f"\n{name} Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return results

In [6]:
# logistic regression trainng 
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Now train with the scaled data
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train_scaled, y_train)

res_lr = evaluate_model("Logistic Regression", lr, X_test_scaled, y_test)




Logistic Regression Confusion Matrix:
[[56854    10]
 [   43    55]]


In [8]:
rf = RandomForestClassifier(n_estimators=100,max_depth=10, random_state=42)
rf.fit(X_train, y_train)

res_rf = evaluate_model("Random Forest", rf, X_test, y_test)


Random Forest Confusion Matrix:
[[56861     3]
 [   25    73]]


In [9]:
# xg boost
xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
xgb.fit(X_train, y_train)

res_xgb = evaluate_model("XGBoost", xgb, X_test, y_test)

c:\Users\Bacha\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:36:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost Confusion Matrix:
[[56859     5]
 [   19    79]]


In [10]:
# isolation forest 
iso = IsolationForest(contamination=0.001)
iso.fit(X_train)

y_pred_iso = iso.predict(X_test)

# Convert: -1 → fraud (1), 1 → normal (0)
y_pred_iso = np.where(y_pred_iso == -1, 1, 0)

results_iso = {
    "Model": "Isolation Forest",
    "Accuracy": accuracy_score(y_test, y_pred_iso),
    "Precision": precision_score(y_test, y_pred_iso),
    "Recall": recall_score(y_test, y_pred_iso),
    "F1": f1_score(y_test, y_pred_iso),
    "ROC-AUC": roc_auc_score(y_test, y_pred_iso)
}

print("\nIsolation Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_iso))

# comparsion tables 
results = [res_lr, res_rf, res_xgb, results_iso]

df_results = pd.DataFrame(results)
df_results


Isolation Forest Confusion Matrix:
[[56832    32]
 [   73    25]]


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.999070,0.846154,0.561224,0.674847,0.975861
1,Random Forest,0.999508,0.960526,0.744898,0.839080,0.972718
2,XGBoost,0.999579,0.940476,0.806122,0.868132,0.940590
3,Isolation Forest,0.998157,0.438596,0.255102,0.322581,0.627270


In [ ]:
# best model 
best_model = rf   # whichever performs best

In [12]:
# sving the best model 
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/fraud_model.pkl")

['../models/fraud_model.pkl']